# ⑫ o2pulse_flattening — `o2p_v3`
⑫ o2pulse_flattening — 3세대 `o2p_v3` (유일 노트북). v2와 다른 점: 프로토콜(램프 속도) 층화 사전지정 민감도 SE8–SE10 추가(사용자 결정 2026-10-22).
SE8 표준 램프만의 P1 · P2 · SE9 램프 3범주별 유병률 · SE10 S2 로지스틱에 램프 범주 보정. 램프 = 마지막 120 s 속도 증가율(10 s 구간 OLS, km/h/min): ≤ 0.98 느림 · 0.98–1.02 표준 · > 1.02 빠름.
(이 범주는 2026-10-22 AI 사후 관찰에서 나왔다 — 원고에 밝힌다.) v2 이전 기록: 2026-10-22 사용자 결정으로 가스 신호 이상 2검사 제외(EXCLUDE_USER).
사전지정: 03_RESEARCH/projects/o2pulse_flattening/analysis.md (2026-10-22 승인 1) · 계획: 05_RUNS/o2pulse_flattening/2026-10-22_plan.md
질문: 18세 이상 선수 최대 트레드밀 검사에서 마지막 120 s O2 pulse(10 s 평균) 기울기 <= 0(Perim 2011)의
      유병률(P1, 봉인군 70% 참가자당 첫 검사, Wilson CI)과 재검사 kappa(P2, 봉인군 재검사 1인 1쌍, 참가자 부트스트랩 CI).
실행: 이 노트북을 자기 폴더(04_OUTPUTS/code_o2p_v3)에서 Restart & Run All. 경로는 환경변수 O2P_RAW · O2P_OUT로 바꿀 수 있다(OUT가 이미 있으면 멈춘다).
통계 방법: [기초통계 p.571–577](비율 구간; Wilson은 Brown 2001) · [기초통계 p.322, p.326](로지스틱) · kappa CI: Klar 2002(책 없음)
해석 점검: [해석학 p.155–161, p.150–153, p.56–58] · 수학 근거: [수학 p.237–240](OR) · 그림: [자료분석 p.262–268] · [시각화 p.311–312]


In [ ]:

import os, sys, json, hashlib, platform, datetime, math
from importlib import metadata
import numpy as np, pandas as pd
import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = None

H = os.getcwd()   # 노트북 폴더(04_OUTPUTS/code_o2p_v3)
V = os.path.normpath(os.path.join(H, '..', '..'))
RAW = os.environ.get('O2P_RAW') or os.path.join(H, '..', 'raw')   # 공개 저장소판: 저장소 루트의 raw/ (PhysioNet 원자료는 저장소에 없음)
OUT = os.environ.get('O2P_OUT') or os.path.join(H, '..', 'results_o2p_v3')
assert not os.path.exists(OUT), f'결과 폴더가 이미 있다(덮어쓰지 않음): {OUT}'
os.makedirs(OUT); FIG = os.path.join(OUT, 'figures'); os.makedirs(FIG)
V2_RES = os.environ.get('O2P_V2') or os.path.join(V, '04_OUTPUTS', 'results_o2p_v2')   # 없으면 v2 대조 생략(공개 저장소판)
# 사용자 결정 2026-10-22 — 곡선 확인으로 측정 오류 판단(원인: 가스 신호). 제외 전후는 v1_vs_v2.csv
EXCLUDE_USER = {'741_1': 'gas_signal_collapse (VO2 1,250 -> 291 mL/min while speed 8 -> 14 km/h and VE rose)',
                '539_1': 'implausibly_low_VO2 (21 mL/kg/min at 17.4 km/h; RER peak 0.91)'}
SCREEN_IDS = os.environ.get('O2P_SCREEN_IDS') or os.path.join(V, '04_OUTPUTS', 'results_malaga_screen_2026-10-21', 'screen_ids.csv')   # 없으면 seed로만 재현
SEED, JSEED, NB = 20261021, 20261022, 2000
SHA = {'subject-info.csv': 'e9ca5ba20eb1272df56c11d3e584ec3997d5484454812bd94966e58082415f16',
       'test_measure.csv': '3b5ed5586dc7412ccdfb26f804e29c06482a2a1df2060399c2731ebc68d4613d'}
PRESPEC = ['P1', 'P2', 'S1', 'S2', 'S3'] + [f'SE{i}' for i in range(1, 11)] + ['O', 'D']
PLAN_IDS = ['P1', 'P2', 'S1', 'S2', 'S3', 'SE1', 'SE2', 'SE3', 'SE4', 'SE5', 'SE6', 'SE7', 'SE8', 'SE9', 'SE10', 'O', 'D']
assert PRESPEC == PLAN_IDS, '사전지정 ↔ plan 불일치'
VALID = {'vo2peak_kg': (15, 90), 'hrmax': (120, 230), 'o2p_max': (5, 45), 'rer_peak': (0.9, 1.6)}
EXPECTED_OUTPUTS = ['run_info.json', 'requirements_exact.txt', 'engine_check.csv', 'sample_flow.csv', 'per_test.csv',
    'P1_prevalence.csv', 'P2_kappa.csv', 'S1_vo2plateau.csv', 'S2_logistic.csv', 'S2_influence.csv', 'S3_definitions.csv',
    'S3_def_kappa.csv', 'sensitivity.csv', 'posthoc.csv', 'v2_vs_v3.csv', 'ramp_strata.csv', 'S2_ramp_adjusted.csv', 'G4_strata.csv', 'outliers.csv', 'decisions.json'] + \
    [f'figures/{g}.png' for g in ['G1_sample_flow', 'G2_engine', 'G3_primary', 'G4_strata', 'G5_robustness', 'G6_factors',
                                  'G7_definitions', 'G8_trajectories', 'G9_influence', 'G10_outliers']] + \
    [f'figures/{g}_data.csv' for g in ['G3a', 'G3b', 'G4', 'G6', 'G8', 'G10']]
Z = stats.norm.ppf(.975)

In [ ]:
# ── 0. 엔진 검증 ─────────────────────────────────────────────
def wilson(k, n):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + Z**2 / n; c = (p + Z**2 / (2 * n)) / d; h = Z * math.sqrt(p * (1 - p) / n + Z**2 / (4 * n * n)) / d
    return (c - h, c + h)
def kappa(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool); po = np.mean(a == b)
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return np.nan if pe >= 1 else (po - pe) / (1 - pe)
eng = []; rng = np.random.default_rng(SEED)
for k, n in [(0, 10), (3, 17), (41, 254), (250, 600), (600, 600)]:
    lo, hi = proportion_confint(k, n, method='wilson'); w = wilson(k, n)
    eng.append(dict(check=f'wilson {k}/{n} lo', a=w[0], b=lo)); eng.append(dict(check=f'wilson {k}/{n} hi', a=w[1], b=hi))
for i in range(5):
    a = rng.random(80) < .3; b = np.where(rng.random(80) < .7, a, rng.random(80) < .3)
    eng.append(dict(check=f'kappa sim{i}', a=kappa(a, b), b=cohen_kappa_score(a, b)))
for i in range(5):
    x = np.sort(rng.random(12) * 2); y = .5 * x + rng.normal(0, .3, 12)
    eng.append(dict(check=f'slope sim{i}', a=np.polyfit(x, y, 1)[0], b=stats.linregress(x, y).slope))
X = rng.normal(size=(600, 3)); yy = (rng.random(600) < 1 / (1 + np.exp(-(X @ [.5, -1, .3] - .4)))).astype(int)
f1 = sm.Logit(yy, sm.add_constant(X)).fit(disp=0); f2 = LogisticRegression(penalty=None, max_iter=10000, tol=1e-10).fit(X, yy)
for j in range(3): eng.append(dict(check=f'logit beta{j+1}', a=f1.params[j + 1], b=f2.coef_[0][j]))
eng.append(dict(check='logit intercept', a=f1.params[0], b=f2.intercept_[0]))
E = pd.DataFrame(eng); E['absdiff'] = (E.a - E.b).abs()
assert (E[~E.check.str.startswith('logit')].absdiff < 1e-10).all() and (E[E.check.str.startswith('logit')].absdiff < 1e-4).all(), E
E.to_csv(os.path.join(OUT, 'engine_check.csv'), index=False)

In [ ]:
# ── 1. 원자료 · 입력 전제 · 봉인 분할 ──────────────────────────
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
for fn, ex in SHA.items(): assert sha(os.path.join(RAW, fn)) == ex, f'SHA-256 불일치 {fn}'
s = pd.read_csv(os.path.join(RAW, 'subject-info.csv'))
m = pd.read_csv(os.path.join(RAW, 'test_measure.csv')).sort_values(['ID_test', 'time'], kind='stable')
assert s.ID_test.is_unique and set(s.Sex.unique()) <= {0, 1} and len(s) == 992 and s.ID.nunique() == 857
assert set(m.ID_test.unique()) == set(s.ID_test) and (m.groupby('ID_test').time.diff().dropna() >= 0).all()
ids = np.sort(s.ID.unique()); S_IDS = set(np.random.default_rng(SEED).choice(ids, size=int(round(.3 * len(ids))), replace=False).tolist())
if os.path.exists(SCREEN_IDS): assert S_IDS == set(pd.read_csv(SCREEN_IDS).screen_ID), '봉인 분할 재현 실패'
else: print(f'screen_ids.csv 없음 → seed {SEED}로 만든 분할을 쓴다(점검군 {len(S_IDS)}명)')
s['sealed'] = ~s.ID.isin(S_IDS)

In [ ]:
# ── 2. 검사별 지표 ─────────────────────────────────────────────
def sm10(y, w=10): return pd.Series(y).rolling(w, min_periods=3, center=True).mean().to_numpy()
def roll30(t, y):
    k = ~np.isnan(y); out = np.full(len(t), np.nan)
    if k.sum() < 3: return out
    out[k] = pd.Series(y[k], index=pd.to_timedelta(t[k], unit='s')).rolling('30s', min_periods=3).mean().to_numpy()
    return out
def window(t, v, h, te, win, bw):
    k = (t >= te - win) & (t < te)
    d = pd.DataFrame(dict(b=np.floor((t[k] - (te - win)) / bw).astype(int), v=v[k], h=h[k], t=t[k])).groupby('b').mean()
    d = d[d.v.notna() & d.h.notna() & (d.h > 0)]
    nb = int(round(win / bw)); need = max(3, math.ceil(2 * nb / 3))
    r = dict(nbins=len(d), evaluable=len(d) >= need)
    if len(d) >= 3:
        x = d.t.to_numpy() / 60; o2p = (d.v / d.h).to_numpy(); lr = stats.linregress(x, o2p)
        r.update(slope=lr.slope, slope_hi=lr.slope + stats.t.ppf(.975, len(d) - 2) * lr.stderr,
                 vo2_slope=np.polyfit(x, d.v.to_numpy(), 1)[0], hr_slope=np.polyfit(x, d.h.to_numpy(), 1)[0])
    return r
VARIANTS = {'main': (120, 10), 'b30': (120, 30), 'w60': (60, 10), 'w180': (180, 10)}
info = s.set_index('ID_test'); rows = []; traj = []
for tid, g in m.groupby('ID_test'):
    g = g[g.HR.between(30, 230) | g.HR.isna()]
    t = (g.time - g.time.min()).to_numpy(float); hr = g.HR.to_numpy(float); sp = g.Speed.to_numpy(float)
    vo2 = g.VO2.to_numpy(float); vco2 = g.VCO2.to_numpy(float)
    r = dict(id_test=tid, subj=int(info.at[tid, 'ID']), k=int(str(tid).split('_')[1]), age=float(info.at[tid, 'Age']),
             sex=int(info.at[tid, 'Sex']), weight=float(info.at[tid, 'Weight']), sealed=bool(info.at[tid, 'sealed']), fail='')
    ip = int(np.nanargmax(sp)); aft = np.where((t > t[ip]) & (sp < np.nanmax(sp) - 2.0))[0]
    if tid == '218_1': r['fail'] = 'speed_channel_anomaly'
    elif tid in EXCLUDE_USER: r['fail'] = 'user_excluded_signal'
    elif len(aft) == 0: r['fail'] = 'no_exercise_end'
    if r['fail']: rows.append(r); continue
    te = t[aft[0]]; ex = t < te; r['t_end'] = te
    r['vo2peak_kg'] = np.nanmax(roll30(t[ex], vo2[ex])) / r['weight']; r['hrmax'] = np.nanmax(sm10(hr[ex]))
    r['phrmax'] = 100 * r['hrmax'] / (208 - .7 * r['age']); r['speed_max'] = float(np.nanmax(sp[ex]))
    kr = (t >= te - 120) & (t < te)   # SE8–SE10: 마지막 120 s 속도 증가율
    dr = pd.DataFrame(dict(b=np.floor((t[kr] - (te - 120)) / 10).astype(int), s=sp[kr], t=t[kr])).groupby('b').mean()
    r['ramp_last120'] = np.polyfit(dr.t / 60, dr.s, 1)[0] if len(dr) >= 3 else np.nan
    with np.errstate(invalid='ignore', divide='ignore'): rer = roll30(t[ex], vco2[ex]) / roll30(t[ex], vo2[ex])
    r['rer_peak'] = float(np.nanmax(rer[np.isfinite(rer)])) if np.isfinite(rer).any() else np.nan
    for nm, (w, b) in VARIANTS.items():
        for kk, vv in window(t, vo2, hr, te, w, b).items(): r[f'{nm}_{kk}'] = vv
    # 운동 전체 구간 지표(S3 · O₂P 최대)
    d10 = pd.DataFrame(dict(b=(t[ex] // 10).astype(int), v=vo2[ex], h=hr[ex], t=t[ex])).groupby('b').mean()
    d10 = d10[d10.v.notna() & d10.h.notna() & (d10.h > 0)]
    o10 = (d10.v / d10.h).to_numpy(); r['o2p_max'] = float(o10.max()) if len(o10) else np.nan
    if len(o10) >= 12:
        r['Ddecl'] = bool((d10.t.to_numpy()[np.argmax(o10)] < .9 * te) and (o10[-1] <= .95 * o10.max()))
    d30 = pd.DataFrame(dict(b=(t[ex] // 30).astype(int), v=vo2[ex], h=hr[ex], t=t[ex])).groupby('b').mean()
    d30 = d30[d30.v.notna() & d30.h.notna() & (d30.h > 0)]
    if len(d30) >= 12:
        tb = d30.t.to_numpy(); o = (d30.v / d30.h).to_numpy(); mid = (tb >= te / 3) & (tb < 2 * te / 3); fin = tb >= 2 * te / 3
        if mid.sum() >= 3 and fin.sum() >= 3:
            smid = np.polyfit(tb[mid] / 60, o[mid], 1)[0]; sfin = np.polyfit(tb[fin] / 60, o[fin], 1)[0]
            r['Dthird'] = bool(sfin <= 0); r['Dratio'] = bool(sfin <= .25 * smid)
    # G8 궤적(주 창, 10 s 구간)
    k = (t >= te - 120) & (t < te)
    if k.sum():
        dd = pd.DataFrame(dict(b=np.floor((t[k] - (te - 120)) / 10).astype(int), v=vo2[k], h=hr[k])).groupby('b').mean().reset_index()
        dd['id_test'] = tid; traj.append(dd)
    rows.append(r)
T = pd.DataFrame(rows)
for nm in VARIANTS:
    T[f'{nm}_F'] = np.where(T[f'{nm}_evaluable'] == True, T[f'{nm}_slope'] <= 0, np.nan)
T['vo2_plateau'] = np.where(T.main_evaluable == True, T.main_vo2_slope <= 0, np.nan)
T['Dci'] = np.where(T.main_evaluable == True, T.main_slope_hi <= 0, np.nan)
T['adult'] = T.age >= 18
T['ramp_cat'] = pd.cut(T.ramp_last120, [-np.inf, .98, 1.02, np.inf], labels=['slow (≤0.98)', 'standard (0.98–1.02]', 'fast (>1.02)'])
T.to_csv(os.path.join(OUT, 'per_test.csv'), index=False)   # 검사별 파생자료 — 로컬 전용(DUA), 공개 금지
TR = pd.concat(traj, ignore_index=True)

def first_per_subj(D):
    return D.sort_values(['subj', 'age', 'k']).drop_duplicates('subj', keep='first').reset_index(drop=True)   # 행 단위로 고른다(열마다 섞이지 않게, 검증 지적 2026-10-22)
def pair_per_subj(D):
    P = []
    for sid, g in D.sort_values(['subj', 'age', 'k']).groupby('subj'):
        if len(g) >= 2: P.append((sid, g.iloc[0], g.iloc[1]))
    return P

In [ ]:
# ── 3. 표본 흐름 ─────────────────────────────────────────────
ok = T.fail == ''
flow = [('tests_total', len(T), T.subj.nunique()),
        ('excl_speed_channel_anomaly', int((T.fail == 'speed_channel_anomaly').sum()), None),
        ('excl_user_signal_741_1_539_1', int((T.fail == 'user_excluded_signal').sum()), None),
        ('excl_no_exercise_end', int((T.fail == 'no_exercise_end').sum()), None),
        ('excl_age_lt18', int((ok & ~T.adult).sum()), None),
        ('excl_not_evaluable_main', int((ok & T.adult & (T.main_evaluable != True)).sum()), None)]
A = T[ok & T.adult & (T.main_evaluable == True)].copy()
flow += [('evaluable_adult', len(A), A.subj.nunique()), ('screen_30pct_set_aside', int((~A.sealed).sum()), A[~A.sealed].subj.nunique()),
         ('sealed_tests', int(A.sealed.sum()), A[A.sealed].subj.nunique())]
P1S = first_per_subj(A[A.sealed]); PAIRS = pair_per_subj(A[A.sealed])
flow += [('P1_sealed_first_tests', len(P1S), P1S.subj.nunique()), ('P2_sealed_pairs', len(PAIRS), len(PAIRS))]
FL = pd.DataFrame(flow, columns=['step', 'tests', 'participants']); FL.to_csv(os.path.join(OUT, 'sample_flow.csv'), index=False)

In [ ]:
# ── 4. P1 · P2 ───────────────────────────────────────────────
def prev(D, col='main_F'):
    x = D[col].dropna().astype(bool); k, n = int(x.sum()), len(x); lo, hi = wilson(k, n)
    return dict(k=k, n=n, p=k / n if n else np.nan, lo=lo, hi=hi)
def kappa_boot(P, col='main_F', nb=NB, seed=SEED):
    P = [(sid, a, b) for sid, a, b in P if pd.notna(a[col]) and pd.notna(b[col])]
    x = np.array([bool(a[col]) for _, a, _ in P]); y = np.array([bool(b[col]) for _, _, b in P]); n = len(P)
    if n < 2: return dict(n_pairs=n)
    kp = kappa(x, y); r = np.random.default_rng(seed); bs = []
    for _ in range(nb):
        i = r.integers(0, n, n); bs.append(kappa(x[i], y[i]))
    bs = np.array(bs); ok_ = ~np.isnan(bs)
    lo, hi = np.percentile(bs[ok_], [2.5, 97.5]) if ok_.sum() else (np.nan, np.nan)
    return dict(n_pairs=n, kappa=kp, lo=lo, hi=hi, agree=float(np.mean(x == y)), both_pos=int((x & y).sum()), both_neg=int((~x & ~y).sum()),
                discordant=int((x != y).sum()), boot_undefined=int((~ok_).sum()), conditional_ci=n < 50)
p1 = prev(P1S); p2 = kappa_boot(PAIRS)
p1['decision'] = '흔하다(하한 ≥ 10%)' if p1['lo'] >= .10 else '흔하다고 말할 수 없음(하한 < 10%)'
p1['vs_perim_20pct'] = '포함' if p1['lo'] <= .20 <= p1['hi'] else ('CI가 20%보다 높음' if p1['lo'] > .20 else 'CI가 20%보다 낮음')
p1['vs_mainzer_35pct'] = '포함' if p1['lo'] <= .35 <= p1['hi'] else ('CI가 35%보다 높음' if p1['lo'] > .35 else 'CI가 35%보다 낮음')
p2['decision'] = ('개인 고유 특성으로 볼 수 없다(상한 < 0.40)' if p2['hi'] < .40 else '재현된다(하한 ≥ 0.40)' if p2['lo'] >= .40 else '판정 불가')
pd.DataFrame([p1]).to_csv(os.path.join(OUT, 'P1_prevalence.csv'), index=False)
pd.DataFrame([p2]).to_csv(os.path.join(OUT, 'P2_kappa.csv'), index=False)

In [ ]:
# ── 5. 이차 ─────────────────────────────────────────────────
pos = P1S[P1S.main_F == True]; neg = P1S[P1S.main_F == False]
k1 = int((pos.vo2_plateau == True).sum()); lo, hi = wilson(k1, len(pos))
k0 = int((neg.vo2_plateau == True).sum()); lo0, hi0 = wilson(k0, len(neg))
S1 = pd.DataFrame([dict(group='F_pos', n=len(pos), vo2_plateau=k1, p=k1 / len(pos), lo=lo, hi=hi, hr_slope_median=pos.main_hr_slope.median(),
                         hr_slope_q1=pos.main_hr_slope.quantile(.25), hr_slope_q3=pos.main_hr_slope.quantile(.75)),
                    dict(group='F_neg', n=len(neg), vo2_plateau=k0, p=k0 / len(neg), lo=lo0, hi=hi0, hr_slope_median=neg.main_hr_slope.median(),
                         hr_slope_q1=neg.main_hr_slope.quantile(.25), hr_slope_q3=neg.main_hr_slope.quantile(.75))])
S1.to_csv(os.path.join(OUT, 'S1_vo2plateau.csv'), index=False)

FAC = [('age10', 'age', 10, 'Age (per 10 y)'), ('sex', 'sex', 1, 'Female (vs male)'), ('vo2kg5', 'vo2peak_kg', 5, 'VO2peak (per 5 mL/kg/min)'),
       ('phr5', 'phrmax', 5, '%pred HRmax (per 5 %-pt)'), ('rer01', 'rer_peak', .1, 'Peak RER (per 0.1)'), ('spd1', 'speed_max', 1, 'Peak speed (per 1 km/h)')]
LD = P1S[['subj', 'main_F'] + [c for _, c, _, _ in FAC]].dropna().copy()
for nm, c, sc, _ in FAC: LD[nm] = LD[c] / sc
Xl = sm.add_constant(LD[[f[0] for f in FAC]]); yl = LD.main_F.astype(int)
fit = sm.Logit(yl, Xl).fit(disp=0)
ci = fit.conf_int(); pv = fit.pvalues[[f[0] for f in FAC]]; holm = multipletests(pv, method='holm')[1]
S2 = pd.DataFrame([dict(term=f[0], label=f[3], OR=np.exp(fit.params[f[0]]), lo=np.exp(ci.loc[f[0], 0]), hi=np.exp(ci.loc[f[0], 1]),
                        p=pv[f[0]], p_holm=holm[i], n=int(fit.nobs), events=int(yl.sum())) for i, f in enumerate(FAC)])
# 영향점(GLM 동일 모형)
glm = sm.GLM(yl, Xl, family=sm.families.Binomial()).fit(); inf = glm.get_influence(); nn = int(glm.nobs); kk = Xl.shape[1] - 1
cook = inf.cooks_distance[0]; lev = inf.hat_matrix_diag; dfb = inf.dfbetas
rp = glm.resid_pearson.to_numpy() / np.sqrt(1 - lev); rd = glm.resid_deviance.to_numpy() / np.sqrt(1 - lev)
rst = np.sign(rd) * np.sqrt(lev * rp**2 + (1 - lev) * rd**2)   # Williams 우도 잔차 = 외부 스튜던트화(삭제) 편차 잔차의 근사 (사전지정 '외부 스튜던트화 편차 잔차')
bcrit = stats.t.ppf(1 - .05 / (2 * nn), nn - kk - 2)
INF = pd.DataFrame(dict(subj=LD.subj.values, leverage=lev, cooks_d=cook, resid_stud=rst, fitted=glm.fittedvalues.values, F=yl.values))
for j, f in enumerate(FAC): INF[f'dfbeta_{f[0]}'] = dfb[:, j + 1]
INF['flag_lev'] = lev > 2 * (kk + 1) / nn; INF['flag_cook'] = cook > 4 / nn; INF['flag_resid'] = np.abs(rst) > bcrit
INF['flag_dfbeta'] = (np.abs(dfb[:, 1:]) > 2 / np.sqrt(nn)).any(1)
keep = ~INF.flag_cook.values; fit2 = sm.Logit(yl[keep], Xl[keep]).fit(disp=0); ci2 = fit2.conf_int()
S2['OR_wo_cook'] = [np.exp(fit2.params[f[0]]) for f in FAC]; S2['lo_wo_cook'] = [np.exp(ci2.loc[f[0], 0]) for f in FAC]
S2['hi_wo_cook'] = [np.exp(ci2.loc[f[0], 1]) for f in FAC]; S2['n_wo_cook'] = int(keep.sum())
S2['p_holm_wo_cook'] = multipletests(fit2.pvalues[[f[0] for f in FAC]], method='holm')[1]
S2.to_csv(os.path.join(OUT, 'S2_logistic.csv'), index=False); INF.to_csv(os.path.join(OUT, 'S2_influence.csv'), index=False)

DEFS = [('main_F', 'D120 (primary)'), ('Dthird', 'Dthird'), ('Dratio', 'Dratio'), ('Ddecl', 'Ddecl'), ('Dci', 'Dci')]
S3 = pd.DataFrame([dict(definition=lab, **prev(P1S, c)) for c, lab in DEFS]); S3.to_csv(os.path.join(OUT, 'S3_definitions.csv'), index=False)
DK = []
for c1, l1 in DEFS:
    for c2, l2 in DEFS:
        d = P1S[[c1, c2]].dropna(); a_, b_ = d[c1].astype(bool).to_numpy(), d[c2].astype(bool).to_numpy(); r_ = np.random.default_rng(SEED)
        bs = [kappa(a_[i], b_[i]) for i in (r_.integers(0, len(d), len(d)) for _ in range(NB))] if c1 != c2 else [1.0]
        DK.append(dict(def1=l1, def2=l2, n=len(d), kappa=kappa(a_, b_), lo=np.nanpercentile(bs, 2.5), hi=np.nanpercentile(bs, 97.5)))
pd.DataFrame(DK).to_csv(os.path.join(OUT, 'S3_def_kappa.csv'), index=False)

In [ ]:
# ── 6. 민감도 ───────────────────────────────────────────────
def subset(col_eval, ages_all=False, sealed_only=True, rer_min=None):
    D = T[ok & (T[col_eval] == True)]
    if not ages_all: D = D[D.adult]
    if sealed_only: D = D[D.sealed]
    if rer_min is not None: D = D[D.rer_peak >= rer_min]
    return D
def cluster_prev(D, col='main_F', nb=NB, seed=SEED):
    D = D[D[col].notna()]; subj = D.subj.unique(); r = np.random.default_rng(seed); g = {u: d[col].astype(bool).to_numpy() for u, d in D.groupby('subj')}
    ps = []
    for _ in range(nb):
        pick = r.choice(subj, len(subj)); x = np.concatenate([g[u] for u in pick]); ps.append(x.mean())
    x = D[col].astype(bool); return dict(k=int(x.sum()), n=len(x), p=x.mean(), lo=np.percentile(ps, 2.5), hi=np.percentile(ps, 97.5))
SEN = []
def add(id_, label, D, col):
    a = prev(first_per_subj(D), col); b = kappa_boot(pair_per_subj(D), col)
    SEN.append(dict(id=id_, label=label, analysis='P1', est=a['p'], lo=a['lo'], hi=a['hi'], n=a['n']))
    SEN.append(dict(id=id_, label=label, analysis='P2', est=b.get('kappa'), lo=b.get('lo'), hi=b.get('hi'), n=b.get('n_pairs')))
add('main', 'Primary (120 s, 10 s bins)', subset('main_evaluable'), 'main_F')
add('SE1', '30 s bins', subset('b30_evaluable'), 'b30_F')
add('SE2', 'Window 60 s', subset('w60_evaluable'), 'w60_F')
add('SE3', 'Window 180 s', subset('w180_evaluable'), 'w180_F')
add('SE4', 'All ages (incl. <18 y)', subset('main_evaluable', ages_all=True), 'main_F')
c5 = cluster_prev(subset('main_evaluable')); SEN.append(dict(id='SE5', label='All sealed tests (cluster CI)', analysis='P1', est=c5['p'], lo=c5['lo'], hi=c5['hi'], n=c5['n']))
SEN.append(dict(id='SE5', label='All sealed tests (cluster CI)', analysis='P2', est=np.nan, lo=np.nan, hi=np.nan, n=np.nan))
add('SE6', '100% data (incl. screen 30%)', subset('main_evaluable', sealed_only=False), 'main_F')
add('SE7', 'Peak RER ≥ 1.05 only', subset('main_evaluable', rer_min=1.05), 'main_F')
# SE8 — 표준 램프만(P1: 봉인군 첫 검사 중 표준, P2: 일차 쌍 중 두 검사 모두 표준)
STD = 'standard (0.98–1.02]'
a8 = prev(P1S[P1S.ramp_cat == STD]); b8 = kappa_boot([pp for pp in PAIRS if pp[1]['ramp_cat'] == STD and pp[2]['ramp_cat'] == STD])
SEN.append(dict(id='SE8', label='Standard ramp only (last 120 s 0.98–1.02 km/h/min)', analysis='P1', est=a8['p'], lo=a8['lo'], hi=a8['hi'], n=a8['n']))
SEN.append(dict(id='SE8', label='Standard ramp only (last 120 s 0.98–1.02 km/h/min)', analysis='P2', est=b8.get('kappa'), lo=b8.get('lo'), hi=b8.get('hi'), n=b8.get('n_pairs')))
SENS = pd.DataFrame(SEN); SENS.to_csv(os.path.join(OUT, 'sensitivity.csv'), index=False)
# SE9 — 램프 3범주별 유병률(기술)
RS = pd.DataFrame([dict(ramp=str(lv), **prev(d)) for lv, d in P1S.groupby('ramp_cat', observed=True)]); RS.to_csv(os.path.join(OUT, 'ramp_strata.csv'), index=False)
# SE10 — S2 로지스틱 + 램프 범주(기준 = 표준)
LDR = LD.merge(P1S[['subj', 'ramp_cat']], on='subj', how='left'); LDR = LDR[LDR.ramp_cat.notna()]
LDR['ramp_slow'] = (LDR.ramp_cat == 'slow (≤0.98)').astype(int); LDR['ramp_fast'] = (LDR.ramp_cat == 'fast (>1.02)').astype(int)
fr10 = sm.Logit(LDR.main_F.astype(int), sm.add_constant(LDR[[f[0] for f in FAC] + ['ramp_slow', 'ramp_fast']])).fit(disp=0); c10 = fr10.conf_int()
h10 = multipletests(fr10.pvalues[[f[0] for f in FAC]], method='holm')[1]
S2R = pd.DataFrame([dict(term=t_, label=lab_, OR=np.exp(fr10.params[t_]), lo=np.exp(c10.loc[t_, 0]), hi=np.exp(c10.loc[t_, 1]), p=fr10.pvalues[t_],
                         p_holm=(h10[i] if i < len(FAC) else np.nan), n=int(fr10.nobs))
                    for i, (t_, lab_) in enumerate([(f[0], f[3]) for f in FAC] + [('ramp_slow', 'Slow ramp (vs standard)'), ('ramp_fast', 'Fast ramp (vs standard)')])])
S2R.to_csv(os.path.join(OUT, 'S2_ramp_adjusted.csv'), index=False)

# 층화 기술(G4)
P1S['agegrp'] = pd.cut(P1S.age, [18, 30, 40, 100], right=False, labels=['18–29', '30–39', '≥40'])
G4 = []
for nm, col in [('sex', 'sex'), ('age', 'agegrp'), ('ramp', 'ramp_cat')]:
    for lv, d in P1S.groupby(col, observed=True):
        q = prev(d); G4.append(dict(stratifier=nm, level=('Female' if lv == 1 else 'Male') if nm == 'sex' else str(lv), **q))
G4D = pd.DataFrame(G4); G4D.to_csv(os.path.join(OUT, 'G4_strata.csv'), index=False)

In [ ]:
# ── 7. 이상치(1회 탐지, 표시만) ─────────────────────────────────
OUTL = []; nP = len(P1S)
for c in ['vo2peak_kg', 'o2p_max', 'main_slope']:
    x = P1S[c]; med = x.median(); mad = (x - med).abs().median(); M = .6745 * (x - med) / mad
    for i in np.where(M.abs() > 3.5)[0]:
        OUTL.append(dict(subj=P1S.subj.iloc[i], id_test=P1S.id_test.iloc[i], variable=c, value=x.iloc[i], rule='robust z |M|>3.5', M=M.iloc[i],
                         expected_false_pos=nP * 2 * stats.norm.sf(3.5), cause='needs_review', handling='표시만(제거 안 함)'))
for c, (a, b) in VALID.items():
    x = P1S[c]
    for i in np.where((x < a) | (x > b))[0]:
        OUTL.append(dict(subj=P1S.subj.iloc[i], id_test=P1S.id_test.iloc[i], variable=c, value=x.iloc[i], rule=f'valid range {a}–{b}', M=np.nan,
                         expected_false_pos=np.nan, cause='needs_review', handling='표시만(제거 안 함)'))
OL = pd.DataFrame(OUTL, columns=['subj', 'id_test', 'variable', 'value', 'rule', 'M', 'expected_false_pos', 'cause', 'handling'])
OL.to_csv(os.path.join(OUT, 'outliers.csv'), index=False)

In [ ]:
# ── 7b. 사후(○) 민감도 — 독립 검증 지적(2026-10-22) 뒤 추가, 사전지정 아님 ──────────
PH = []; base = subset('main_evaluable')
rer_bad = set(T[(T.rer_peak < VALID['rer_peak'][0]) | (T.rer_peak > VALID['rer_peak'][1])].id_test)
for id_, lab, D in [('PH1', 'Excl. peak RER outside 0.9–1.6', base[~base.id_test.isin(rer_bad)]),
                    ('PH2', 'Excl. all flagged outlier candidates', base[~base.id_test.isin(set(OL.id_test))])]:
    a = prev(first_per_subj(D)); b = kappa_boot(pair_per_subj(D))
    PH.append(dict(id=id_, label=lab, analysis='P1', est=a['p'], lo=a['lo'], hi=a['hi'], n=a['n']))
    PH.append(dict(id=id_, label=lab, analysis='P2', est=b.get('kappa'), lo=b.get('lo'), hi=b.get('hi'), n=b.get('n_pairs')))
LDr = LD[(LD.rer_peak >= VALID['rer_peak'][0]) & (LD.rer_peak <= VALID['rer_peak'][1])]
fr = sm.Logit(LDr.main_F.astype(int), sm.add_constant(LDr[[f[0] for f in FAC]])).fit(disp=0); hr_ = multipletests(fr.pvalues[[f[0] for f in FAC]], method='holm')[1]
cr = fr.conf_int(); j = [f[0] for f in FAC].index('rer01')
PH.append(dict(id='PH1', label='Excl. peak RER outside 0.9–1.6', analysis='S2 rer01 OR', est=np.exp(fr.params['rer01']), lo=np.exp(cr.loc['rer01', 0]),
               hi=np.exp(cr.loc['rer01', 1]), n=int(fr.nobs), p_holm=hr_[j]))
PHD = pd.DataFrame(PH); PHD['marker'] = '○ post hoc'; PHD.to_csv(os.path.join(OUT, 'posthoc.csv'), index=False)

In [ ]:
# ── 8. 그림 G1–G10 — figure_show_all_n.md §7(묶음) · §8(양식: ggplot2 theme_classic 재현, 산점도·히트맵 제외) ──────────
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
plt.rcParams.update({'font.size': 9, 'font.family': 'DejaVu Sans', 'axes.edgecolor': '#222222', 'axes.linewidth': .8,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
BLUE, ORANGE, GREY, NAVY = '#2a78d6', '#eb6834', '#8a8a8a', '#1b4f9c'
jr = np.random.default_rng(JSEED)
def gg(ax, grid='y'):
    ax.set_axisbelow(True)
    if grid: ax.grid(axis=grid, color='#ececec', lw=.35)
def ttl(ax, t, sub=None):
    ax.set_title(t, loc='left', fontweight='bold', fontsize=9.5, pad=14 if sub else 6)
    if sub: ax.text(0, 1.01, sub, transform=ax.transAxes, fontsize=7.5, color='#666666', va='bottom')
def save(fig, name):
    fig.savefig(os.path.join(FIG, name + '.png'), dpi=200, bbox_inches='tight')
    if display is not None: display(fig)
    plt.close(fig)
def quasi(y, width=.32):
    y = np.asarray(y, float); k = ~np.isnan(y); out = np.zeros(len(y))
    if k.sum() < 3: return out
    kde = stats.gaussian_kde(y[k]); d = kde(y[k]); d = d / d.max()
    order = np.argsort(np.argsort(y[k])); vdc = np.array([int(bin(i)[2:][::-1], 2) / 2 ** len(bin(i)[2:]) for i in range(1, k.sum() + 1)])
    out[k] = (vdc[order] - .5) * 2 * width * d; return out
def violin_panel(ax, groups, labels, colors_pts, mean_ci=True):
    """바이올린 + quasirandom 점 + ◆ 평균(95% CI) + 수치. groups = 값 배열 목록, colors_pts = 점 색 배열 목록."""
    for j, (y, c) in enumerate(zip(groups, colors_pts)):
        y = np.asarray(y, float); vp = ax.violinplot(y[~np.isnan(y)], [j], widths=.8, showextrema=False)
        for b in vp['bodies']: b.set_facecolor('#dfe7f2'); b.set_edgecolor('none'); b.set_alpha(1)
        ax.scatter(j + quasi(y), y, s=4, c=c, lw=0, alpha=.8, zorder=3)
        if mean_ci:
            mu = np.nanmean(y); se = np.nanstd(y, ddof=1) / np.sqrt(np.sum(~np.isnan(y)))
            ax.errorbar([j + .45], [mu], yerr=[[Z * se]], fmt='D', color='k', ms=4, capsize=2, zorder=4)
            ax.text(j + .45, mu + 3 * Z * se + .15, f'{mu:.2f} [{mu - Z*se:.2f}, {mu + Z*se:.2f}]', fontsize=6.5, ha='center', va='bottom', bbox=dict(fc='white', ec='none', pad=.5))
    ax.set_xticks(range(len(groups)), labels); ax.set_xlim(-.6, len(groups) + .05)
def logx(ax, ticks=(.5, 1, 2)):
    ax.set_xscale('log'); ax.xaxis.set_major_locator(FixedLocator(ticks)); ax.xaxis.set_major_formatter(FixedFormatter([f'{t:g}' for t in ticks]))
    ax.xaxis.set_minor_locator(NullLocator())

In [ ]:
# G1
fig, ax = plt.subplots(figsize=(7, 4.2)); ax.axis('off'); y = .9
for _, r in FL.iterrows():
    lab = r.step.replace('_', ' ').replace('lt18', '< 18 y').replace('30pct', '30%'); unit = 'pairs' if r.step.startswith('P2') else 'tests'
    txt = f'{lab}: {r.tests} {unit}' + (f' / {int(r.participants)} participants' if pd.notna(r.participants) and not r.step.startswith('P2') else '')
    ex_ = r.step.startswith('excl'); ax.text(.08 if ex_ else .02, y, ('− ' if ex_ else '') + txt, fontsize=9, transform=ax.transAxes, color='#666666' if ex_ else 'k'); y -= .085
ttl(ax, 'G1 Sample flow', 'sealed 70% = confirmatory set; screen 30% set aside (seen during feasibility screening)'); save(fig, 'G1_sample_flow')

In [ ]:
# G2
fig, ax = plt.subplots(figsize=(4.4, 4)); gg(ax, 'both'); ax.scatter(E.b, E.a, s=14, color=NAVY); lim = [min(E.a.min(), E.b.min()), max(E.a.max(), E.b.max())]
ax.plot(lim, lim, color=GREY, lw=.8); ax.set_xlabel('Reference implementation'); ax.set_ylabel('This script')
ttl(ax, 'G2 Engine checks', f'Wilson · κ · OLS slope · logistic, n={len(E)}; max |diff| = {E.absdiff.max():.1e}'); save(fig, 'G2_engine')

In [ ]:
# G3
fig, axs = plt.subplots(1, 3, figsize=(12, 4.2), gridspec_kw=dict(width_ratios=[1.5, .7, 1.2], wspace=.45))
ax = axs[0]; gg(ax); grp = [P1S[P1S.sex == 0], P1S[P1S.sex == 1]]
violin_panel(ax, [d.main_slope for d in grp], [f'Male\nn={len(grp[0])}', f'Female\nn={len(grp[1])}'],
             [np.where(d.main_F == True, ORANGE, BLUE) for d in grp])
ax.axhline(0, color='#222222', lw=.7, ls='--'); ax.set_ylabel('O2 pulse slope, last 120 s (mL/beat per min)')
ttl(ax, 'a  P1 raw data', 'every analysed test; orange = slope ≤ 0 (flattening); ◆ mean (95% CI)')
G3a = pd.DataFrame(dict(subj=pd.concat([d.subj for d in grp]), sex=['Male'] * len(grp[0]) + ['Female'] * len(grp[1]),
                        slope=pd.concat([d.main_slope for d in grp]), F=pd.concat([d.main_F for d in grp])))
assert len(G3a) == p1['n'] == len(P1S); G3a.to_csv(os.path.join(FIG, 'G3a_data.csv'), index=False)
ax = axs[1]; gg(ax); ax.errorbar([0], [100 * p1['p']], yerr=[[100 * (p1['p'] - p1['lo'])], [100 * (p1['hi'] - p1['p'])]], fmt='o', color='k', capsize=4)
ax.text(.08, 100 * p1['p'], f" {100*p1['p']:.1f}%\n [{100*p1['lo']:.1f}, {100*p1['hi']:.1f}]", fontsize=7, va='center')
for v, lab, c in [(35, 'Mainzer 2026', GREY), (20, 'Perim 2011', GREY), (10, 'decision 10%', ORANGE)]:
    ax.axhline(v, color=c, lw=.7, ls=':' if c == GREY else '--'); ax.text(-.45, v + .8, lab, fontsize=6.5, color=c)
ax.set_xlim(-.5, .9); ax.set_xticks([0], [f'n={p1["n"]}']); ax.set_ylim(0, 60); ax.set_ylabel('Prevalence (%)')
ttl(ax, 'b  P1 estimate', 'Wilson 95% CI')
ax = axs[2]; gg(ax)
y1 = np.array([a['main_slope'] for _, a, _ in PAIRS]); y2 = np.array([b['main_slope'] for _, _, b in PAIRS])
f1_ = np.array([bool(a['main_F']) for _, a, _ in PAIRS]); f2_ = np.array([bool(b['main_F']) for _, _, b in PAIRS])
for a_, b_, c_ in zip(y1, y2, f1_ != f2_): ax.plot([0, 1], [a_, b_], color=ORANGE if c_ else '#c9c9c9', lw=.7, alpha=.9)
ax.scatter(np.zeros(len(y1)), y1, s=6, color=np.where(f1_, ORANGE, BLUE), zorder=3); ax.scatter(np.ones(len(y2)), y2, s=6, color=np.where(f2_, ORANGE, BLUE), zorder=3)
ax.axhline(0, color='#222222', lw=.7, ls='--'); ax.set_xticks([0, 1], ['Test 1', 'Test 2']); ax.set_xlim(-.3, 1.3); ax.set_ylabel('O2 pulse slope (mL/beat per min)')
ttl(ax, f'c  P2 repeat pairs, n={p2["n_pairs"]}', f'κ {p2["kappa"]:.2f} [{p2["lo"]:.2f}, {p2["hi"]:.2f}]; orange line = class changed ({p2["discordant"]} pairs)')
G3b = pd.DataFrame(dict(subj=[s_ for s_, _, _ in PAIRS], slope1=y1, slope2=y2, F1=f1_, F2=f2_)); assert len(G3b) == p2['n_pairs']; G3b.to_csv(os.path.join(FIG, 'G3b_data.csv'), index=False)
fig.suptitle('G3 Primary results — sealed 70%', x=.01, y=1.04, ha='left', fontweight='bold'); save(fig, 'G3_primary')

In [ ]:
# G4
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True, gridspec_kw=dict(width_ratios=[2, 3, 3])); g4 = []
for ax, nm, col, levels in [(axs[0], 'sex', 'sex', [(0, 'Male'), (1, 'Female')]), (axs[1], 'age', 'agegrp', [(l, l) for l in ['18–29', '30–39', '≥40']]),
                            (axs[2], 'ramp', 'ramp_cat', [(l, l) for l in ['slow (≤0.98)', 'standard (0.98–1.02]', 'fast (>1.02)']])]:
    gg(ax)
    for j, (lv, lab) in enumerate(levels):
        d = P1S[P1S[col] == lv]; q = G4D[(G4D.stratifier == nm) & (G4D.level == lab)].iloc[0]
        ax.scatter(j - .2 + jr.uniform(-.08, .08, len(d)), 100 * d.main_F.astype(float) + jr.uniform(-3, 3, len(d)), s=3,
                   color=np.where(d.main_F == True, ORANGE, BLUE), alpha=.5, lw=0)
        ax.errorbar([j + .15], [100 * q.p], yerr=[[100 * (q.p - q.lo)], [100 * (q.hi - q.p)]], fmt='o', color='k', capsize=3)
        ax.text(j + .2, 100 * q.p, f' {100*q.p:.1f}%', fontsize=7, va='center')
        g4 += [dict(stratifier=nm, level=lab, subj=u, F=f) for u, f in zip(d.subj, d.main_F)]
    ax.set_xticks(range(len(levels)), [f'{lab}\nn={int((P1S[col] == lv).sum())}' for lv, lab in levels]); ttl(ax, {'sex': 'Sex', 'age': 'Age group', 'ramp': 'Ramp, last 120 s (km/h/min) — SE9'}[nm])
axs[0].set_ylabel('Flattening (%)'); fig.suptitle('G4 Prevalence by stratum (descriptive, no interaction test)', x=.01, y=1.06, ha='left', fontweight='bold')
fig.text(.01, .985, 'points = each person at 0 or 100 (jittered); ● % with Wilson 95% CI', fontsize=7.5, color='#666666')
G4P = pd.DataFrame(g4); assert len(G4P) == 3 * len(P1S) - P1S.agegrp.isna().sum() - P1S.ramp_cat.isna().sum(); G4P.to_csv(os.path.join(FIG, 'G4_data.csv'), index=False); save(fig, 'G4_strata')

In [ ]:
# G5
SG = pd.concat([SENS.assign(marker='●'), PHD[PHD.analysis.isin(['P1', 'P2'])].assign(marker='○')], ignore_index=True)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True, gridspec_kw=dict(wspace=.08))
for ax, an, xl in [(axs[0], 'P1', 'P1 prevalence (%)'), (axs[1], 'P2', 'P2 Cohen κ')]:
    gg(ax, 'x'); d = SG[SG.analysis == an].reset_index(drop=True); sc = 100 if an == 'P1' else 1
    for i, r in d.iterrows():
        if pd.isna(r.est): ax.text(0 if an == 'P2' else 30, -i, 'not applicable', fontsize=7, va='center', color='#666666'); continue
        hollow = r.marker == '○'
        ax.errorbar([sc * r.est], [-i], xerr=[[sc * (r.est - r.lo)], [sc * (r.hi - r.est)]], fmt='o', mfc='white' if hollow else 'k', mec='k', color=GREY if hollow else 'k', ms=4, capsize=2)
        ax.text(sc * r.hi, -i + .2, f' n={int(r.n)}', fontsize=6.5, color='#666666')
    if an == 'P1': ax.set_yticks(-np.arange(len(d)), [f'{m} {l}' for m, l in zip(d.marker, d.label)], fontsize=7.5)
    ax.set_xlabel(xl); ax.axvline(10 if an == 'P1' else .40, color=ORANGE, lw=.7, ls='--')
fig.suptitle('G5 Robustness — ● pre-specified, ○ post hoc (added after independent check); n = tests (P1) or pairs (P2)', x=.01, y=1.0, ha='left', fontweight='bold', fontsize=9); save(fig, 'G5_robustness')

In [ ]:
# G6
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw=dict(width_ratios=[1, 1.4], wspace=.35))
ax = axs[0]; gg(ax, 'x')
for i, r in S2.iterrows():
    ax.errorbar([r.OR], [-i + .12], xerr=[[r.OR - r.lo], [r.hi - r.OR]], fmt='o', color='k', ms=4, capsize=2)
    r2 = S2R[S2R.term == r.term].iloc[0]; ax.errorbar([r2.OR], [-i - .12], xerr=[[r2.OR - r2.lo], [r2.hi - r2.OR]], fmt='s', color=GREY, ms=4, capsize=2)
logx(ax, (.5, 1, 2)); ax.axvline(1, color=GREY, lw=.8)
ax.set_yticks(-np.arange(len(S2)), [f'{l}\nP_Holm ' + ('< 0.001' if p < .001 else f'= {p:.3f}') for l, p in zip(S2.label, S2.p_holm)], fontsize=7)
ax.set_xlabel('Odds ratio (95% CI), log scale'); ttl(ax, 'a  S2 adjusted logistic', f'● S2 (n={int(fit.nobs)}, events={int(yl.sum())}) · ■ + ramp category, SE10 (n={int(fr10.nobs)}); P_Holm = S2')
ax = axs[1]; gg(ax); g6 = []
for j, (nm, c, sc_, lab) in enumerate(FAC):
    for fv, col in [(0, BLUE), (1, ORANGE)]:
        v = LD[LD.main_F.astype(int) == fv][c]; z = (v - LD[c].mean()) / LD[c].std()
        ax.scatter(j + (-.2 if fv == 0 else .2) + quasi(z.to_numpy(), .12), z, s=3, color=col, alpha=.6, lw=0)
        g6 += [dict(factor=nm, F=fv, value=x_) for x_ in v]
ax.set_xticks(range(len(FAC)), ['Age', 'Female', 'VO2peak', '%pred HRmax', 'Peak RER', 'Peak speed'], fontsize=7); ax.set_ylabel('Standardized value (z)')
ttl(ax, 'b  Raw factor values', 'blue = no flattening, orange = flattening (female coded 0/1)')
G6P = pd.DataFrame(g6); assert len(G6P) == len(FAC) * len(LD); G6P.to_csv(os.path.join(FIG, 'G6_data.csv'), index=False)
fig.suptitle('G6 Associated factors (● S2 pre-specified · ■ SE10 adjusted for ramp category; ramp ORs in S2_ramp_adjusted.csv)', x=.01, y=1.04, ha='left', fontweight='bold'); save(fig, 'G6_factors')

In [ ]:
# G7 (히트맵 대신 점 + CI, §8)
DKD = pd.DataFrame(DK); fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.8), gridspec_kw=dict(wspace=.6))
ax = axs[0]; gg(ax, 'x')
for i, r in S3.iterrows():
    ax.errorbar([100 * r.p], [-i], xerr=[[100 * (r.p - r.lo)], [100 * (r.hi - r.p)]], fmt='o', color='k', capsize=2); ax.text(100 * r.hi, -i + .2, f' {100*r.p:.1f}%', fontsize=6.5)
ax.set_yticks(-np.arange(len(S3)), [f'{d} (n={n})' for d, n in zip(S3.definition, S3.n)], fontsize=8); ax.set_xlabel('Prevalence (%), Wilson 95% CI'); ttl(ax, 'a  Prevalence by definition')
ax = axs[1]; gg(ax, 'x'); dd = DKD[(DKD.def1 == 'D120 (primary)') & (DKD.def2 != 'D120 (primary)')].reset_index(drop=True)
for i, r in dd.iterrows(): ax.errorbar([r.kappa], [-i], xerr=[[r.kappa - r.lo], [r.hi - r.kappa]], fmt='o', color='k', capsize=2)
ax.set_yticks(-np.arange(len(dd)), dd.def2, fontsize=8); ax.set_xlim(0, 1); ax.set_xlabel('Cohen κ with the primary definition (bootstrap 95% CI)')
ttl(ax, 'b  Agreement with D120', f'same {len(P1S)} tests'); fig.suptitle('G7 Definition dependence (S3)', x=.01, y=1.04, ha='left', fontweight='bold'); save(fig, 'G7_definitions')

In [ ]:
# G8
TT = TR[TR.id_test.isin(P1S.id_test)].merge(P1S[['id_test', 'main_F']], on='id_test'); TT['o2p'] = TT.v / TT.h; TT['tsec'] = TT.b * 10 + 5 - 120
fig, axs = plt.subplots(1, 3, figsize=(11.5, 3.6), gridspec_kw=dict(wspace=.35))
for ax, col, lab in [(axs[0], 'o2p', 'O2 pulse (mL/beat)'), (axs[1], 'v', 'VO2 (mL/min)'), (axs[2], 'h', 'HR (bpm)')]:
    gg(ax)
    for tid, d in TT.groupby('id_test'): ax.plot(d.tsec, d[col], color='#e3e3e3', lw=.3)
    for fv, cc, lb in [(False, BLUE, f'no flattening (n={len(neg)})'), (True, ORANGE, f'flattening (n={len(pos)})')]:
        g_ = TT[TT.main_F == fv].groupby('tsec')[col]; mm, se = g_.mean(), g_.sem()
        ax.fill_between(mm.index, mm - Z * se, mm + Z * se, color=cc, alpha=.25, lw=0); ax.plot(mm.index, mm.values, color=cc, lw=1.8, label=lb)
    ax.set_xlabel('Time before exercise end (s)'); ax.set_ylabel(lab)
axs[1].legend(loc='upper center', bbox_to_anchor=(.5, -.2), ncol=2, fontsize=7.5)
ttl(axs[0], 'a  O2 pulse'); ttl(axs[1], 'b  VO2', f'VO2 plateau: flattening {100*S1.p[0]:.0f}% [{100*S1.lo[0]:.0f}, {100*S1.hi[0]:.0f}] vs no {100*S1.p[1]:.0f}% [{100*S1.lo[1]:.0f}, {100*S1.hi[1]:.0f}]'); ttl(axs[2], 'c  HR')
assert TT.id_test.nunique() == len(P1S); TT.to_csv(os.path.join(FIG, 'G8_data.csv'), index=False)
fig.suptitle('G8 Last 120 s trajectories (grey = each test; lines = group mean with 95% CI)', x=.01, y=1.06, ha='left', fontweight='bold'); save(fig, 'G8_trajectories')

In [ ]:
# G9
fig, axs = plt.subplots(1, 4, figsize=(14, 3.6), gridspec_kw=dict(wspace=.4))
ax = axs[0]; gg(ax, 'both'); ax.scatter(INF.fitted, INF.resid_stud, s=5, color=NAVY, alpha=.6); [ax.axhline(v, color=ORANGE, ls='--', lw=.6) for v in (bcrit, -bcrit)]
ax.set_xlabel('Fitted probability'); ax.set_ylabel('Likelihood (ext. studentized) residual'); ttl(ax, f'a  Residuals, n={nn}', f'Bonferroni ±{bcrit:.2f}: {int(INF.flag_resid.sum())} beyond')
ax = axs[1]; gg(ax, 'both'); (osm, osr), (sl_, ic_, _) = stats.probplot(INF.resid_stud, dist='norm'); ax.scatter(osm, osr, s=5, color=NAVY); ax.plot(osm, sl_ * osm + ic_, color=GREY, lw=.8)
ax.set_xlabel('Normal quantile'); ax.set_ylabel('Residual quantile'); ttl(ax, 'b  Q–Q', 'binary outcome: two bands expected')
ax = axs[2]; gg(ax); ax.vlines(np.arange(nn), 0, INF.cooks_d, color=NAVY, lw=.6); ax.axhline(4 / nn, color=ORANGE, ls='--', lw=.6)
ax.set_xlabel('Observation'); ax.set_ylabel("Cook's D"); ttl(ax, "c  Cook's D", f'> 4/n: {int(INF.flag_cook.sum())} · leverage > 2(k+1)/n: {int(INF.flag_lev.sum())}')
ax = axs[3]; gg(ax, 'x')
for i, r in S2.iterrows():
    ax.errorbar([r.OR], [-i + .12], xerr=[[r.OR - r.lo], [r.hi - r.OR]], fmt='o', color='k', ms=3, capsize=2)
    ax.errorbar([r.OR_wo_cook], [-i - .12], xerr=[[r.OR_wo_cook - r.lo_wo_cook], [r.hi_wo_cook - r.OR_wo_cook]], fmt='s', color=GREY, ms=3, capsize=2)
logx(ax, (.5, 1, 2)); ax.axvline(1, color=GREY, lw=.8); ax.set_yticks(-np.arange(len(S2)), ['Age', 'Female', 'VO2peak', '%pred HRmax', 'Peak RER', 'Peak speed'], fontsize=7)
ttl(ax, 'd  OR before/after', f'● all n={nn} · ■ without Cook > 4/n n={int(keep.sum())}')
fig.suptitle('G9 S2 residual and influence diagnostics', x=.01, y=1.06, ha='left', fontweight='bold'); save(fig, 'G9_influence')

In [ ]:
# G10
fig, axs = plt.subplots(1, 5, figsize=(16, 3.8), gridspec_kw=dict(width_ratios=[1, 1, 1, 1, 1.3], wspace=.55)); g10 = []
for ax, c, lab in zip(axs[:4], ['vo2peak_kg', 'o2p_max', 'main_slope', 'rer_peak'], ['VO2peak (mL/kg/min)', 'O2 pulse max (mL/beat)', 'O2P slope last 120 s', 'Peak RER']):
    gg(ax); x = P1S[c].to_numpy(float); vp = ax.violinplot(x[~np.isnan(x)], [0], widths=.8, showextrema=False)
    for b in vp['bodies']: b.set_facecolor('#dfe7f2'); b.set_alpha(1)
    xx = quasi(x); ax.scatter(xx, x, s=4, color=GREY, alpha=.7, lw=0)
    cand = set(OL[(OL.variable == c) & (OL.rule.str.startswith('robust'))].id_test); rng_ = set(OL[(OL.variable == c) & (OL.rule.str.startswith('valid'))].id_test)
    mk = P1S.id_test.isin(cand).values; mv = P1S.id_test.isin(rng_).values
    ax.scatter(xx[mk], x[mk], marker='x', color=ORANGE, s=25, zorder=4); ax.scatter(xx[mv], x[mv], marker='D', facecolor='none', edgecolor='k', s=25, zorder=4)
    if c in VALID:
        for v in VALID[c]: ax.axhline(v, color=GREY, lw=.6, ls=':')
    ax.set_xticks([]); ttl(ax, lab, f'n={np.sum(~np.isnan(x))} · × {mk.sum()} · ◇ {mv.sum()}')
    g10 += [dict(variable=c, id_test=t_, value=v_) for t_, v_ in zip(P1S.id_test, x)]
ax = axs[4]; gg(ax, 'x'); rows_ = [('All (primary)', SENS[(SENS.id == 'main') & (SENS.analysis == 'P1')].iloc[0], '●')] + \
    [({'PH1': 'Excl. RER\noutside range', 'PH2': 'Excl. all\ncandidates'}[r.id], r, '○') for _, r in PHD[PHD.analysis == 'P1'].iterrows()]
for i, (lab, r, mk_) in enumerate(rows_):
    ax.errorbar([100 * r.est], [-i], xerr=[[100 * (r.est - r.lo)], [100 * (r.hi - r.est)]], fmt='o', mfc='white' if mk_ == '○' else 'k', mec='k', color='k', capsize=2)
    ax.text(100 * r.hi, -i + .2, f' n={int(r.n)}', fontsize=6.5, color='#666666')
ax.set_yticks(-np.arange(len(rows_)), [f'{m} {l}' for l, _, m in rows_], fontsize=7); ax.set_xlabel('P1 prevalence (%)'); ttl(ax, 'P1 before/after exclusion', '○ post hoc; candidates are not removed in the primary analysis')
G10P = pd.DataFrame(g10); assert len(G10P) == 4 * len(P1S); G10P.to_csv(os.path.join(FIG, 'G10_data.csv'), index=False)
fig.suptitle('G10 Outlier screen (× robust z |M| > 3.5, ◇ outside valid range; flagged only)', x=.01, y=1.08, ha='left', fontweight='bold'); save(fig, 'G10_outliers')

In [ ]:
# ── 9. 판정 · 실행 정보 · 산출물 assert ─────────────────────────
dec = dict(P1=dict(prevalence=p1['p'], lo=p1['lo'], hi=p1['hi'], n=p1['n'], decision=p1['decision'], vs_perim_20=p1['vs_perim_20pct'], vs_mainzer_35=p1['vs_mainzer_35pct']),
           P2=dict(kappa=p2['kappa'], lo=p2['lo'], hi=p2['hi'], n_pairs=p2['n_pairs'], agreement=p2['agree'], decision=p2['decision'], conditional_ci=p2['conditional_ci']))
json.dump(dec, open(os.path.join(OUT, 'decisions.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=float)
pk = ['numpy', 'pandas', 'statsmodels', 'scipy', 'scikit-learn', 'matplotlib']
open(os.path.join(OUT, 'requirements_exact.txt'), 'w').write('\n'.join(f'{p}=={metadata.version(p)}' for p in pk) + f'\n# python {platform.python_version()}\n')
json.dump(dict(generation='o2p_v3', excluded_user=EXCLUDE_USER, run_at=datetime.datetime.now().isoformat(timespec='seconds'), seed=SEED, jitter_seed=JSEED, n_boot=NB,
               raw=RAW, sha256=SHA, sha_algorithm='SHA-256', screen_ids=os.path.relpath(SCREEN_IDS, V), python=platform.python_version(),
               platform=platform.platform(), flow=FL.to_dict('records')), open(os.path.join(OUT, 'run_info.json'), 'w', encoding='utf-8'),
          ensure_ascii=False, indent=1, default=str)
# v2 대비 — 일차 · 이차는 바뀌면 안 된다(추가 민감도만). 공개 저장소판: results_o2p_v2가 없으면 생략
if os.path.isdir(V2_RES):
    cmp_ = []
    for f, cols in [('P1_prevalence.csv', ['k', 'n', 'p', 'lo', 'hi']), ('P2_kappa.csv', ['n_pairs', 'kappa', 'lo', 'hi', 'agree']), ('S1_vo2plateau.csv', ['n', 'vo2_plateau', 'p'])]:
        a_ = pd.read_csv(os.path.join(V2_RES, f)); b_ = pd.read_csv(os.path.join(OUT, f))
        for i in range(len(a_)):
            for c in cols: cmp_.append(dict(file=f, row=i, stat=c, v2=a_[c][i], v3=b_[c][i]))
    a_ = pd.read_csv(os.path.join(V2_RES, 'S2_logistic.csv')).set_index('term'); b_ = pd.read_csv(os.path.join(OUT, 'S2_logistic.csv')).set_index('term')
    for t_ in a_.index:
        for c in ['OR', 'lo', 'hi', 'p_holm']: cmp_.append(dict(file='S2_logistic.csv', row=t_, stat=c, v2=a_.at[t_, c], v3=b_.at[t_, c]))
    CMP = pd.DataFrame(cmp_); CMP['diff'] = CMP.v3 - CMP.v2; CMP.to_csv(os.path.join(OUT, 'v2_vs_v3.csv'), index=False)
    assert (CMP['diff'].abs() < 1e-9).all(), CMP[CMP['diff'].abs() >= 1e-9]
else:
    pd.DataFrame([dict(file='(results_o2p_v2 없음 — 대조 생략)', row='', stat='', v2=np.nan, v3=np.nan, diff=0.0)]).to_csv(os.path.join(OUT, 'v2_vs_v3.csv'), index=False)
missing = [f for f in EXPECTED_OUTPUTS if not os.path.exists(os.path.join(OUT, f))]
assert not missing, f'정본 산출물 누락: {missing}'
print(json.dumps(dec, ensure_ascii=False, indent=1, default=float))
print(FL.to_string(index=False)); print(S1.round(3).to_string(index=False)); print(S2.round(3).to_string(index=False)); print(S3.round(3).to_string(index=False))
print(RS.round(3).to_string(index=False)); print(S2R.round(3).to_string(index=False)); print(SENS.round(3).to_string(index=False)); print('outlier rows', len(OL), OL.groupby(['variable', 'rule']).size().to_dict()); print(G4D.round(3).to_string(index=False))